In [14]:
store_df = df[df["Store"] == 1].copy()
store_df = store_df.groupby("Date")["Revenue"].sum().reset_index()
store_df = store_df.sort_values("Date")
store_df["Revenue"] = store_df["Revenue"].astype(float)
store_df = store_df.reset_index(drop=True)
store_df.head()

,Date,Revenue
0,2010-02-05,1643690.90
1,2010-02-12,1641957.44
2,2010-02-19,1611968.17
3,2010-02-26,1409727.59
4,2010-03-05,1554806.68


In [15]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error

series = store_df["Revenue"]

model = ARIMA(series, order=(3,1,3))
fit = model.fit()

store_df["forecast"] = fit.predict(start=1, end=len(series))

aligned = store_df.dropna(subset=["forecast", "Revenue"])
aligned = aligned[aligned["Revenue"] > 0]
aligned["forecast"] = aligned["forecast"].clip(lower=0)

mape = mean_absolute_percentage_error(
    aligned["Revenue"],
    aligned["forecast"]
)
mape


0.06691619621581295

In [16]:
aligned["forecast"].shape, aligned["Revenue"].shape, mape


((142,), (142,), 0.06691619621581295)

In [17]:
aligned.to_csv("../data/arima_store_1.csv", index=False)